Como iremos fazer um treinamento de um mmodelo ML iremos utilizar o formato ELT

In [2]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [3]:
import os
print(os.environ.get("JAVA_HOME"))

/usr/lib/jvm/java-17-openjdk-amd64


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("cybersecurity_ingestion") \
    .getOrCreate()



df_incedents = spark.read.parquet( "../data/bronze/incidents_master.parquet")
df_market_impact = spark.read.parquet( "../data/bronze/market_impact.parquet")
df_financial_impact = spark.read.parquet( "../data/bronze/financial_impact (1).parquet")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 20:46:31 WARN Utils: Your hostname, carlos-Legion-Slim-5-16IRH8, resolves to a loopback address: 127.0.1.1; using 192.168.1.34 instead (on interface wlp0s20f3)
26/04/02 20:46:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 20:46:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


vou fazer analises gerais em cada dataset e depois paratir para um analise mais especifica de cada 1

In [5]:

df_incedents.show(5)

26/04/02 20:46:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------+--------------------+-------------------+----------+----------------+------------------+--------------+-----------------+------------+-------------+-----------------------+--------------+---------------+---------------------+-----------------------+--------------------+----------------+----------------------+------------------------+---------+--------------------+--------------+--------------------+---------------------+----------------+---------------+-------------+-------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+
|  incident_id|        company_name|company_revenue_usd|country_hq|industry_primary|industry_secondary|employee_count|is_public_company|stock_ticker|incident_date|incident_date_estimated|discovery_date|disclosure_date|attack_vector_primary|attack_vector_secondary|        attack_chain|attributed_group|attribution_confidence|data_compromised_records|d

In [6]:
df_incedents.select("country_hq").distinct().show()

+----------+
|country_hq|
+----------+
|        FI|
|        NL|
|        PL|
|        MX|
|        AT|
|        CZ|
|        PT|
|        HK|
|        TW|
|        CL|
|        AU|
|        CA|
|        GB|
|        BR|
|        DE|
|        ES|
|        IL|
|        ZA|
|        KR|
|        US|
+----------+
only showing top 20 rows


In [7]:
print("Colunas nulas do DataFrame:")
print(df_incedents.columns)

print("\nSchema do DataFrame:")
df_incedents.printSchema()

print("\nContagem de valores nulos por coluna:")
from pyspark.sql.functions import count, when, isnull
df_incedents.select([count(when(isnull(c), c)).alias(c) for c in df_incedents.columns]).show()

Colunas nulas do DataFrame:
['incident_id', 'company_name', 'company_revenue_usd', 'country_hq', 'industry_primary', 'industry_secondary', 'employee_count', 'is_public_company', 'stock_ticker', 'incident_date', 'incident_date_estimated', 'discovery_date', 'disclosure_date', 'attack_vector_primary', 'attack_vector_secondary', 'attack_chain', 'attributed_group', 'attribution_confidence', 'data_compromised_records', 'data_type', 'systems_affected', 'downtime_hours', 'data_source_primary', 'data_source_secondary', 'data_source_type', 'confidence_tier', 'quality_score', 'quality_grade', 'review_flag', 'notes', 'created_at', 'updated_at', 'ingestion_timestamp', 'source_file', 'row_count', 'file_hash']

Schema do DataFrame:
root
 |-- incident_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- company_revenue_usd: double (nullable = true)
 |-- country_hq: string (nullable = true)
 |-- industry_primary: string (nullable = true)
 |-- industry_secondary: string (nullabl

In [8]:
print(f"Total de linhas: {df_incedents.count()}")
print(f"Número de colunas: {len(df_incedents.columns)}")
print(f"Schema: {df_incedents.printSchema()}")

Total de linhas: 850
Número de colunas: 36
root
 |-- incident_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- company_revenue_usd: double (nullable = true)
 |-- country_hq: string (nullable = true)
 |-- industry_primary: string (nullable = true)
 |-- industry_secondary: string (nullable = true)
 |-- employee_count: long (nullable = true)
 |-- is_public_company: boolean (nullable = true)
 |-- stock_ticker: string (nullable = true)
 |-- incident_date: string (nullable = true)
 |-- incident_date_estimated: boolean (nullable = true)
 |-- discovery_date: string (nullable = true)
 |-- disclosure_date: string (nullable = true)
 |-- attack_vector_primary: string (nullable = true)
 |-- attack_vector_secondary: string (nullable = true)
 |-- attack_chain: string (nullable = true)
 |-- attributed_group: string (nullable = true)
 |-- attribution_confidence: string (nullable = true)
 |-- data_compromised_records: double (nullable = true)
 |-- data_type: string (nullable

In [9]:
total_rows = df_incedents.count()
unique_rows = df_incedents.distinct().count()
duplicatas = total_rows - unique_rows
print(f"Total de linhas: {total_rows}")
print(f"Linhas únicas: {unique_rows}")
print(f"Linhas duplicadas: {duplicatas}")
 

Total de linhas: 850
Linhas únicas: 850
Linhas duplicadas: 0


In [10]:
total_registros = df_incedents.count()
incident_ids_unicos = df_incedents.select('incident_id').dropDuplicates().count()
incident_ids_duplicados = total_registros - incident_ids_unicos

print(f"Total de registros: {total_registros}")
print(f"incident_id únicos: {incident_ids_unicos}")
print(f"Registros com incident_id duplicado: {incident_ids_duplicados}")

Total de registros: 850
incident_id únicos: 850
Registros com incident_id duplicado: 0
